# Negotiate, Then Solve: training the agent adapter (4B, two T4s)

The agent-loop rebuild (`src/nts`, docs/agentic.md) runs Qwen3.5-4B in a loop of tool calls. This
notebook trains one LoRA adapter, `nts-agent-v1` by default, on judged episodes exported from telemetry
(`python -m nts export --out data/agent-v1 --models gemini`): good episodes only, in the chat format with
tools. It is `train_multigpu_kaggle.ipynb` (exp branch, 3f347b7) with these changes:

* **Data:** one example per episode, rendered once with the tools by the model's own chat template
  and thinking off (the way llama-server renders a request with `--jinja`), with the loss on its
  assistant turns only (`<tool_call>` blocks or the final text). Each turn's request is a prefix of
  the whole episode, so with causal attention this learns exactly what one example per turn would,
  but reads each conversation once instead of once per turn (about 3x fewer tokens). The
  learning-curve order is stratified by scenario family instead of task.
* **Train:** a plain `Trainer` whose loss asks the model for logits on the learned positions only
  (`logits_to_keep`): the 248k-word output layer over a whole episode would take GBs, and about 5%
  of the tokens are learned.
* **Attention in blocks** (`chunked_attention` in `/tmp/agent_train.py`): on a T4, SDPA has no fused kernel
  that trains with Qwen3.5's head_dim 256, so it kept each full-attention layer's whole score matrix
  (1.5 GB at 5k tokens, which ran a GPU out of memory); blocks of 1024 queries, recomputed in the
  backward pass, against only the keys they can see. Same outputs and gradients as SDPA (checked to
  1e-7), less memory and about half the attention work. The linear-attention layers use
  flash-linear-attention's kernels (`FAST_KERNELS`), not transformers' slow PyTorch fallback.
* **Both GPUs compute** (`PARALLEL = "data"`): in fp16 the model (~8.5 GB) fits on one T4, so the
  training runs as `/tmp/agent_train.py` under `torchrun`, a copy of the model per GPU, each reading
  other episodes of the stream, gradients averaged: about twice the speed of splitting the model
  across the GPUs, where only one computes at a time. If that run fails (out of memory), the
  notebook trains split instead, in the same session. `PARALLEL = "model"` always splits (fp32
  needs it: 17 GB).
* **Settings:** a new adapter, no merge (the server applies adapters), long examples (the tool
  schemas alone are ~2.2k tokens), batch 1, fp16 (the v6 run showed Qwen3.5 trains in it on a T4).
* **Quick check:** greedy next turn on validation turns: the same tool call (name and arguments)
  or, for a final answer, any text?

**Before running**

1. Zip the export folder (`sft-train.jsonl`, `sft-val.jsonl`) and upload it as a Kaggle dataset with
   *agent* in its name; add it (*Add Input*).
2. *Settings*: **Accelerator: GPU T4 x2**, **Internet: on**.
3. *Save Version* > *Save & Run All (Commit)* (`TRIAL = True` runs 20 steps and prints seconds per
   step and peak memory, if a new setting needs checking first). The first steps print loss and peak memory.
5. Serve the GGUF beside the other adapters and evaluate: `python -m nts eval --model llama --adapter nts-agent-v1`.

What follows is the original notebook's description of the training setup (with the model split
across the GPUs; `PARALLEL` above changes that):


**One adapter per task.** The local model is the base Qwen3.5-4B plus a LoRA adapter per group of
tasks (`adapters.toml`, `src/language/adapters.py`), not one model fine-tuned on everything. This
notebook trains one adapter, `ADAPTER`, on the uploaded rows of its `TASKS`, and exports it as a
small LoRA GGUF (tens of MB) that `llama-server` applies to the base GGUF it already serves. So:

* changing one task (new policy rows, a new intent) retrains that task's adapter only, on a
  fraction of the data (reading requests is about half the tokens, the policy rows a quarter);
* `INIT_FROM` updates an adapter instead of training it again: it starts from the earlier
  version's weights and reads only the rows that version has not seen (`trained.json` lists
  them), with `REPLAY` times as many it has seen so it does not forget them;
* the export converts the adapter, it does not merge and quantize the whole model (`MERGE = True`
  still does, for a server without adapters);
* a domain (another institute's handbook) is one more adapter beside the shared ones.

A `TRIAL` run exports its 20-step adapter too: load it on the server (below) to check the
conversion and the serving before committing hours.

LoRA fine-tuning of **Qwen3.5-4B** with plain Hugging Face (transformers, PEFT,
TRL), no Unsloth. The data, the prompt rendering and the quick check are the same as
`train_compiler_kaggle.ipynb`, so the model drops into `--local` the same way.

**Why not Unsloth.** A T4 has no bf16 and Qwen3.5 does not train in fp16, so the
weights stay in float32: about 17 GB for the 4B model, more than one T4 (15 GB).
Unsloth trains on one GPU only, and ordinary multi-GPU training (DDP) copies the whole
model to each GPU, so neither fits. Here `device_map="auto"` splits the layers across
both T4s (about 8 GB each). Only one GPU computes at a time, so the second one adds
memory, not speed: expect roughly 2-3x the time of the 2B run.

**Before running**

1. Upload `data/multitask-readable/` (or `data/multitask-readable.zip`) as a Kaggle dataset with
   *multitask* in its name and add it (*Add Input*). Remove older versions of it.
2. *Settings*: **Accelerator: GPU T4 x2**, **Internet: on**.
3. Run with `TRIAL = True` (20 steps). The training cell prints the seconds per step
   and the projected length of the full run, against Kaggle's 12-hour session limit.
4. Full run: set `TRIAL = False`, then *Save Version* > *Save & Run All (Commit)*.
   A commit runs in the background for up to 12 hours and keeps `/kaggle/working`.
   Training saves a checkpoint every 50 steps and stops itself (with a checkpoint)
   after `TIME_BUDGET_H` hours. If it stopped early, add that version's output as an
   input and commit again: it resumes from the latest checkpoint (`RESUME = True`).

**Learning curve.** One run answers "how much data does the model need": the training
stream is ordered so that every prefix of it has the same mix of tasks as the whole, the
trainer reads it in that order, and the adapter is saved after each fraction in `CURVE`
of what it reads (`READ`: how much of the `EPOCHS` x train-set stream to read). With a
curve the learning rate is constant after warm-up, so the points differ only in how much
was read. The export makes the `CURVE_QUANTS` GGUFs for each point; score each one with the
`--local` evaluations. With `EPOCHS = 1`, the point at 25% has seen a quarter of the
examples once: the curve over unique data. With more epochs it is the curve over reading.

**Fewer examples.** `TRAIN_EXAMPLES = 1000` trains on 1000 of the uploaded examples, each
task in proportion (a prefix of the same stratified order), for `EPOCHS` epochs; `None`
uses them all.

**fp16.** `PRECISION = "fp16"` keeps the frozen weights in fp16 and the adapters in
float32 (mixed precision): about half the memory and several times faster on a T4.
Qwen3.5's linear-attention layers may overflow in fp16; the training cell stops if the
loss is not finite. Try it in a TRIAL before committing hours to it.

In [ ]:
%%capture
FAST_KERNELS = True   # also install flash-linear-attention: Triton kernels for the linear-attention layers (they run on a T4)
# Keep Kaggle's CUDA build of torch: a dependency must not swap it for another build.
!pip freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/torch-pin.txt
!pip install --upgrade -c /tmp/torch-pin.txt "transformers>=5" peft trl accelerate datasets
# Kaggle ships torchao 0.10; PEFT refuses torchao < 0.16 when it adds LoRA layers. Nothing here uses it.
!pip uninstall -y -q torchao
if FAST_KERNELS:
    !pip install -c /tmp/torch-pin.txt flash-linear-attention

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, transformers, peft, trl

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, (
    "Qwen3.5 needs transformers v5. Restart the kernel (Run > Restart) and run from this cell.")
N_GPU = torch.cuda.device_count()
assert N_GPU > 0, (
    f"no GPU (torch {torch.__version__}, CUDA {torch.version.cuda}): this session runs on Kaggle's CPU image. "
    "Set Settings > Accelerator > GPU T4 x2 for this notebook (and check it again in Save Version). "
    "If it is set, another GPU session (e.g. the 2B run) may be using your GPU: wait for it to finish.")
for i in range(N_GPU):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB")
if N_GPU < 2:
    print("WARNING: one GPU. Set Accelerator: GPU T4 x2; the 4B model in float32 does not fit on one T4.")

## Settings

In [ ]:
import glob, json, math, re, time
from pathlib import Path

SESSION_START = time.time()  # Kaggle's 12 hours count from here, not from training

MODEL = "Qwen/Qwen3.5-4B"
# What to train: one LoRA adapter on some tasks of the uploaded rows (adapters.toml).
ADAPTER = "nts-agent-v1"    # its name: the GGUF file and the adapter name the server lists
TASKS = None                # agent rows have no task: every judged-good episode
# Update an earlier version instead of training from the base: its output folder (attach that notebook
# output or a dataset: the folder with trained.json, and the adapter in lora/ or beside it). It reads the rows that version has not seen and REPLAY
# times as many that it has. None: a new adapter. "auto": the one attached input that has a trained.json.
# This run: a new adapter from the base, on `python -m nts export` output.
INIT_FROM = None
REPLAY = 2.0                # twice the new rows again from the ones it has seen, so it keeps them
EXPECT_NEW = None           # with INIT_FROM: about how many rows the data adds; far more means the attached version read other data: stop
MERGE = False               # the server applies adapters per request (--lora ... --lora-init-without-apply)
TRIAL = False               # True: 20 steps, time projection (the training cell prints peak memory for the first steps)
PRECISION = "fp16"          # "fp16" (T4 tensor cores: several times faster) or "fp32" (if the TRIAL stops on a non-finite loss)
EPOCHS = 3
LR = 2e-4
MAX_LEN = 12288             # a whole episode: tool schemas (~2.2k tokens) + the conversation + its last turn (longer: skipped)
BATCH, ACCUM = 1, 8         # 8 episodes per step (about 25 turns); episodes are long, so one at a time
WARMUP = 3                  # optimizer steps
PARALLEL = "data"           # "data": a copy of the model per GPU, both compute (fp16 only); "model": split across them
SEED = 3407
# Weights per GPU (the model is ~17 GB in float32). The output layer is tied to the
# embeddings, so it sits on GPU 0, and the logits (248k vocabulary: ~1.1 GB for the
# longest example, a few GB with the loss) are computed there: keep GPU 0 lighter. If the
# load cell reports CPU offload, raise these; if GPU 0 runs out of memory, move 1-2 GB to GPU 1.
MAX_MEMORY = {0: "8GiB", 1: "12GiB"}
TIME_BUDGET_H = 10.75       # hours since the session started: stop with a checkpoint before Kaggle's 12
RESUME = True               # continue from the latest checkpoint in an attached earlier output
# Learning curve (see the top): how much of the EPOCHS x train-set stream to read, and after
# which fractions of that to save the adapter. CURVE = [] trains as before (shuffled, cosine).
TRAIN_EXAMPLES = None       # train on this many examples (each task in proportion); None: all of them
READ = 1.0
CURVE = [1/3, 2/3, 1.0]     # an adapter after each of the 3 epochs: score all three, and later runs use the fewest epochs that do as well
                            # ([] trains as before: shuffled, cosine)
CURVE_QUANTS = ["Q8_0"]  # the GGUFs made for each curve point (3 files, about 14 GB; the output limit is 20 GB)

found = sorted(glob.glob("/kaggle/input/**/sft-train.jsonl", recursive=True))
assert found, "no sft-train.jsonl under /kaggle/input: add the agent dataset (python -m nts export output)"
agent = [f for f in found if "agent" in f.lower()]
if not agent and len(found) > 1:
    print("WARNING: no dataset path contains 'agent'; using", found[0], "of", found)
DATA = Path((agent or found)[0]).parent
NAME = ADAPTER
OUT = Path("/kaggle/working") / NAME
if INIT_FROM == "auto":
    earlier = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/trained.json", recursive=True)})
    assert len(earlier) == 1, (f"INIT_FROM = 'auto' needs exactly one attached output with a trained.json, found "
                               f"{earlier or 'none'}: attach the earlier version's output, or name its folder")
    INIT_FROM = earlier[0]
if INIT_FROM:
    INIT_FROM = Path(INIT_FROM)
    if not (INIT_FROM / "trained.json").exists():  # a notebook output's or an input's root: find the adapter in it
        INIT_FROM = Path(next(glob.iglob(f"{INIT_FROM}/**/trained.json", recursive=True))).parent
    INIT_LORA = INIT_FROM / "lora" if (INIT_FROM / "lora").exists() else INIT_FROM
    print("updating", json.loads((INIT_FROM / "manifest.json").read_text())["adapter"], "from", INIT_FROM)
print("data:", DATA, "| output:", OUT)

In [ ]:
%%writefile /tmp/agent_train.py
"""The training run of train_agent_kaggle.ipynb, written to /tmp/agent_train.py by the notebook.

    torchrun --nproc_per_node 2 /tmp/agent_train.py run.json    # data parallel: a copy of the model per GPU
    agent_train.train(C, load_model(C, "auto", ...), ...)      # in the notebook: the model split across the GPUs

Data parallel needs the model to fit on one GPU (fp16: ~8.5 GB on a 15 GB T4) and keeps both GPUs computing:
each reads other episodes of the stream, in its order, and the gradients are averaged. Split, only one GPU
computes at a time. ``run.json`` holds the notebook's settings; the stream and the validation episodes are
on disk (``datasets.save_to_disk``); the result (steps, losses, seconds per step) goes to ``result.json``.
"""
import json
import math
import os
import sys
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_from_disk
from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import SequentialSampler
from transformers import AutoModelForImageTextToText, Trainer, TrainerCallback, TrainingArguments

TARGETS = r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
CHUNK = 1024  # queries per block of the chunked attention
# fp16 loss scale at the start. The default (65536) overflows on these episodes and halves once per skipped step:
# the first run skipped 7 steps and settled at 512 (halving once more later). From a halving below that, the
# first steps already update the adapter.
INIT_SCALE = 2.0 ** 8


def chunked_attention(module, query, key, value, attention_mask, scaling=None, dropout=0.0, **kwargs):
    """Causal attention a block of queries at a time, each block recomputed in the backward pass.

    On a T4, SDPA has no fused kernel that trains with Qwen3.5's head_dim 256 (flash attention needs sm80), so it
    falls back to the math kernel and keeps the whole score matrix: heads x n x n in fp32, 1.5 GB per layer
    for a 5k-token episode, which ran one GPU out of memory. Here a block holds CHUNK x n scores at a time, and
    only the keys its queries can see (half the work of the full matrix). query (B, H, L, D), key and value
    (B, H_kv, S, D); with a KV cache (generation) the queries are the last L of S positions."""
    from torch.utils.checkpoint import checkpoint

    rep = query.shape[1] // key.shape[1]
    if rep > 1:
        key, value = key.repeat_interleave(rep, 1), value.repeat_interleave(rep, 1)
    L, S = query.shape[2], key.shape[2]
    scale = scaling if scaling is not None else query.shape[-1] ** -0.5
    pos_k = torch.arange(S, device=query.device)

    def block(q, k, v, allowed):
        s = torch.matmul(q, k.transpose(-1, -2)).float() * scale
        s = s.masked_fill(~allowed, float("-inf"))
        return torch.matmul(torch.softmax(s, -1).to(v.dtype), v)

    out = []
    for i in range(0, L, CHUNK):
        j = min(i + CHUNK, L)
        last = S - L + j  # keys beyond the block's last query are never visible
        allowed = pos_k[None, :last] <= (S - L + torch.arange(i, j, device=query.device))[:, None]
        if attention_mask is not None:  # a 4D mask from transformers: bool (True = attend) or additive
            m = attention_mask[:, :, i:j, :last]
            allowed = allowed & (m if m.dtype == torch.bool else m == 0)
        args = (query[:, :, i:j], key[:, :, :last], value[:, :, :last], allowed)
        out.append(checkpoint(block, *args, use_reentrant=False) if torch.is_grad_enabled() else block(*args))
    return torch.cat(out, 2).transpose(1, 2).contiguous(), None


def register_attention():
    from transformers import AttentionInterface
    from transformers.masking_utils import AttentionMaskInterface, sdpa_mask
    AttentionInterface.register("nts_chunked", chunked_attention)
    AttentionMaskInterface.register("nts_chunked", sdpa_mask)


def load_model(C, device_map, max_memory=None, adapter=None, trainable=True):
    """The base model with LoRA on the language layers: a new adapter, or ``adapter`` (a saved one)."""
    dtype = torch.float32 if C["PRECISION"] == "fp32" else torch.float16
    register_attention()
    model = AutoModelForImageTextToText.from_pretrained(C["MODEL"], dtype=dtype, device_map=device_map,
                                                        max_memory=max_memory, attn_implementation="nts_chunked")
    model.config.use_cache = False
    if adapter:
        return PeftModel.from_pretrained(model, str(adapter), is_trainable=trainable)
    return get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
                                            task_type="CAUSAL_LM", target_modules=TARGETS))


def collator(pad):
    def collate(batch):
        """Right-padded input_ids, attention_mask and labels (-100 on padding)."""
        n = max(len(x["input_ids"]) for x in batch)
        ids = torch.tensor([x["input_ids"] + [pad] * (n - len(x["input_ids"])) for x in batch])
        labels = torch.tensor([x["labels"] + [-100] * (n - len(x["labels"])) for x in batch])
        mask = torch.tensor([[1] * len(x["input_ids"]) + [0] * (n - len(x["input_ids"])) for x in batch])
        return {"input_ids": ids, "attention_mask": mask, "labels": labels}
    return collate


class Episodes(Trainer):
    """The loss on the learned tokens only, with logits for those positions only: the output layer (248k
    vocabulary) over a whole episode would take GBs in fp32, and about 5% of the tokens are learned. The loss
    is the mean over each episode's learned tokens."""
    full_logits = False  # set if the model does not take positions in logits_to_keep

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        target = labels[:, 1:]                                    # position p predicts token p + 1
        # a list, not a tensor: accelerate's hooks move tensor arguments to GPU 0, but the output layer may sit on
        # another GPU, and a tensor index must be on the indexed tensor's device
        keep = (target != -100).any(0).nonzero().squeeze(1).tolist()
        if not Episodes.full_logits:
            try:
                out = model(**inputs, logits_to_keep=keep, use_cache=False)
                logits, target = out.logits, target[:, keep]
            except TypeError:
                Episodes.full_logits = True
                print("WARNING: the model takes no positions in logits_to_keep: full logits (more memory)")
        if Episodes.full_logits:
            out = model(**inputs, use_cache=False)
            logits = out.logits[:, :-1]
        target = target.to(logits.device)
        loss = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), target.reshape(-1), ignore_index=-100)
        return (loss, out) if return_outputs else loss


class InOrder(Episodes):
    """Reads the stream in the order built in the data cell (a Trainer shuffles by default), so the adapter
    saved after a fraction of it has seen that fraction of every task. With several GPUs, accelerate deals
    the batches out in turn, so together they still read the stream in order."""
    def _get_train_sampler(self, *args, **kwargs):
        return SequentialSampler(self.train_dataset)


class Curve(TrainerCallback):
    """Saves the adapter at each curve point, apart from the rotating checkpoints."""
    def __init__(self, C, tok):
        self.C, self.tok, self.steps = C, tok, {int(k): v for k, v in C["CURVE_STEPS"].items()}

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step in self.steps and state.is_world_process_zero:
            f = self.steps[state.global_step]
            path = Path(self.C["OUT"]) / "curve" / f"read-{round(f * 100):03d}"
            model.save_pretrained(str(path))
            self.tok.save_pretrained(str(path))
            (path / "read.json").write_text(json.dumps({
                "fraction": f, "step": state.global_step, "examples": state.global_step * self.C["PER_STEP"],
                "epochs": self.C["EPOCHS"], "read": self.C["READ"], "loss": next(
                    (h["loss"] for h in reversed(state.log_history) if "loss" in h), None)}))
            print(f"curve: adapter after {f:.0%} of the reading saved to {path}")


class Clock(TrainerCallback):
    """Seconds per optimizer step, a stop with a checkpoint before the session limit, a stop if the loss is
    not finite or no step updates the adapter (fp16 overflow), and peak memory for the first steps."""
    def __init__(self, C):
        self.C, self.marks, self.skipped = C, [], 0

    def on_step_end(self, args, state, control, **kwargs):
        self.marks.append(time.time())
        if not self.C["TRIAL"] and (time.time() - self.C["SESSION_START"]) / 3600 > self.C["TIME_BUDGET_H"]:
            print(f"time budget reached at step {state.global_step}: saving a checkpoint and stopping")
            control.should_save = control.should_training_stop = True

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and not math.isfinite(logs["loss"]):
            print("loss is not finite (fp16 overflows here): use PRECISION = 'fp32'")
            control.should_training_stop = True
        if logs and "grad_norm" in logs:  # a skipped step (fp16 overflow) logs a nan gradient norm
            self.skipped = 0 if math.isfinite(logs["grad_norm"]) else self.skipped + 1
            if self.skipped * args.logging_steps >= 10:
                print(f"no update in the last {self.skipped * args.logging_steps} steps (the fp16 loss scale keeps "
                      "overflowing): lower INIT_SCALE or use PRECISION = 'fp32'")
                control.should_training_stop = True
        if (logs and "loss" in logs and state.global_step <= 10 and torch.cuda.is_available()
                and state.is_world_process_zero):
            print(f"step {state.global_step}: loss {logs['loss']:.3f}, " + ", ".join(
                f"GPU {i} peak {torch.cuda.max_memory_allocated(i) / 2**30:.1f} GB" for i in range(torch.cuda.device_count())))

    def per_step(self):
        m = self.marks[5:] if len(self.marks) > 8 else self.marks  # skip warm-up steps
        return (m[-1] - m[0]) / max(len(m) - 1, 1)


def train(C, model, tok, stream, val, processes=1):
    """Trains ``model`` on ``stream``; returns the result the notebook reads (also in OUT/result.json)."""
    model.enable_input_require_grads()  # gradient checkpointing with frozen embeddings
    clock = Clock(C)
    curve = bool(C["CURVE_STEPS"])
    args = dict(
        output_dir = str(Path(C["OUT"]) / "checkpoints"),
        remove_unused_columns = False,        # the collator reads input_ids and labels itself
        prediction_loss_only = True,          # evaluation: the loss only, no logits gathered
        per_device_train_batch_size = C["BATCH"],
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = C["ACCUM"] // processes,  # the same episodes per step on any number of GPUs
        num_train_epochs = 1 if curve else C["EPOCHS"],  # the curve stream already holds the epochs
        max_steps = 20 if C["TRIAL"] else -1,
        learning_rate = C["LR"],
        lr_scheduler_type = "constant_with_warmup" if curve else "cosine",  # comparable curve points
        warmup_steps = C["WARMUP"],
        weight_decay = 0.01,
        optim = "adamw_torch",                # only the adapters train: the optimizer state is small
        fp16 = C["PRECISION"] == "fp16",
        bf16 = False,                         # T4 has no bf16
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        ddp_find_unused_parameters = False,   # every LoRA layer is used (the vision tower has none)
        logging_steps = 2,
        eval_strategy = "no" if C["TRIAL"] else "steps",
        eval_steps = C["STEPS_PER_EPOCH"],
        save_strategy = "no" if C["TRIAL"] else "steps",
        save_steps = 25,
        save_total_limit = 2,
        report_to = "none",
        seed = C["SEED"],
    )
    args.update(C.get("ARGS") or {})
    trainer = (InOrder if curve else Episodes)(
        model=model, processing_class=tok, train_dataset=stream, eval_dataset=val,
        data_collator=collator(C["PAD"]), callbacks=[clock, Curve(C, tok)], args=TrainingArguments(**args))
    trainer.model_accepts_loss_kwargs = False  # compute_loss returns a mean: the Trainer divides by the accumulation
    if trainer.accelerator.scaler is not None:  # fp16; read when train() prepares the optimizer, a resume loads its state
        trainer.accelerator.scaler = torch.amp.GradScaler("cuda", init_scale=C.get("INIT_SCALE", INIT_SCALE))
    main = trainer.is_world_process_zero()
    if main:
        print(f"{processes} process(es); model parallel across GPUs: {trainer.is_model_parallel}; "
              f"{C['BATCH']} x {args['gradient_accumulation_steps']} x {processes} episodes per step")
    if curve:  # the trainer must read the stream in order: the first batch is the stream's first example
        first = next(iter(trainer.get_train_dataloader()))["input_ids"][0].tolist()
        want = trainer.train_dataset[0]["input_ids"]
        assert not main or first[: len(want)] == want, "the trainer shuffles the stream: the curve points would not be prefixes"
    stats = trainer.train(resume_from_checkpoint=C.get("RESUME_FROM"))
    done = trainer.state.global_step >= trainer.state.max_steps
    result = {
        "processes": processes, "steps": trainer.state.global_step, "max_steps": trainer.state.max_steps,
        "done": done, "minutes": stats.metrics["train_runtime"] / 60, "per_step": clock.per_step(),
        "losses": [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h],
        "validation": [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h],
        "peak_gb": [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(torch.cuda.device_count())]}
    if done or C["TRIAL"]:  # the adapter the quick check and the export read
        trainer.save_model(str(Path(C["OUT"]) / "lora"))
    if main:
        (Path(C["OUT"]) / "result.json").write_text(json.dumps(result))
    return result


def main(path):
    C = json.loads(Path(path).read_text())
    rank, world = int(os.environ.get("LOCAL_RANK", 0)), int(os.environ.get("WORLD_SIZE", 1))
    if torch.cuda.is_available():
        torch.cuda.set_device(rank)
        device_map = {"": rank}
    else:
        device_map = {"": "cpu"}
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(C["MODEL"])
    model = load_model(C, device_map, adapter=C.get("INIT_LORA"))
    if rank == 0:
        model.print_trainable_parameters()
    train(C, model, tok, load_from_disk(C["STREAM"]), load_from_disk(C["VAL"]), processes=world)


if __name__ == "__main__":
    main(sys.argv[1])


## Model and LoRA adapter

Qwen3.5 is a vision-language model (`Qwen3_5ForConditionalGeneration`). The whole
model is loaded, so the merged checkpoint keeps its architecture for the GGUF
conversion, but LoRA goes on the language layers only (the same modules as the 2B
run) and the vision tower stays frozen and unused. With `INIT_FROM` the earlier adapter is
loaded, trainable, instead of a new one. With `PARALLEL = "data"` the training processes load it, not
this cell.

In [ ]:
import sys
from transformers import AutoTokenizer

sys.path.insert(0, "/tmp")
import agent_train  # written by the cell above

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.chat_template is None:  # some releases keep the template with the processor
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODEL).chat_template
print("eos:", repr(tok.eos_token))

# data parallel: each training process loads its own copy of the model on its GPU (the Train cell); none here
DATA_PARALLEL = PARALLEL == "data" and N_GPU > 1 and PRECISION == "fp16"
model = None
if not DATA_PARALLEL:
    model = agent_train.load_model({"MODEL": MODEL, "PRECISION": PRECISION}, "auto", MAX_MEMORY,
                                   adapter=INIT_LORA if INIT_FROM else None)
    placed = {str(d) for d in model.hf_device_map.values()}
    print("devices:", sorted(placed))
    assert not placed & {"cpu", "disk"}, (
        "part of the model was offloaded to the CPU (very slow): raise MAX_MEMORY or use PRECISION = 'fp16'")
    for i in range(N_GPU):
        print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GB of weights")
    model.print_trainable_parameters()
print("training:", f"data parallel, a copy of the model on each of {N_GPU} GPUs" if DATA_PARALLEL else
      "one process, the model split across the GPUs")


## Data

One example per exported episode: the whole conversation rendered with the tools by the chat template,
**thinking disabled**, then tokenized once; the labels are the tokens of its assistant turns (-100 elsewhere).
Each turn's request (the conversation before it with a generation prompt, as llama-server renders it) must
be a prefix of the whole rendering; a turn whose request is not (a template that renders past turns
differently) is left unlearned and counted. Episodes are known by a hash of their messages for `INIT_FROM`.


In [ ]:
import json, hashlib, random
from bisect import bisect_right
from collections import Counter, defaultdict
from datasets import Dataset

def render(messages, tools, generation):
    return tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=generation,
                                   enable_thinking=False)

def row_hash(row):
    return hashlib.sha1(json.dumps(row["messages"], sort_keys=True).encode()).hexdigest()[:16]

SKIPPED = Counter()
END = "<|im_end|>"
PLACEHOLDER = {"role": "tool", "name": "end", "content": "{}"}  # after an episode's last turn: keeps it a current
                                                                # turn (a user message would make it history, and
                                                                # Qwen3.5's template then drops its empty <think>)

def turn_spans(row):
    """The whole episode rendered once (followed by PLACEHOLDER), and the character span of each assistant
    turn in it. Turn i's request (the conversation before it, rendered as llama-server renders it, as checked
    against /apply-template: 51/51 turns) must be a prefix of the whole; a turn whose request is not is
    skipped and counted."""
    msgs, tools = row["messages"], row["tools"]
    full = render(msgs + [PLACEHOLDER], tools, False)
    spans = []
    for i, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        prompt = render(msgs[:i], tools, True)
        if not full.startswith(prompt) or END not in full[len(prompt):]:
            SKIPPED["not a prefix"] += 1
            continue
        spans.append((prompt, len(prompt), full.index(END, len(prompt)) + len(END)))
    return full, spans

def episode_example(row):
    """One example per episode: its tokens, with labels on the assistant turns only (-100 elsewhere).
    Every turn's request is a prefix of the episode, so with causal attention this learns what one example
    per turn would, but reads the conversation once instead of once per turn."""
    full, spans = turn_spans(row)
    if not spans:
        return None
    enc = tok(full, return_offsets_mapping=True, add_special_tokens=False)
    ids, starts = enc["input_ids"], [a for _, a, _ in spans]
    labels = []
    for t, (s, _) in zip(ids, enc["offset_mapping"]):
        k = bisect_right(starts, s) - 1
        labels.append(t if k >= 0 and s < spans[k][2] else -100)
    if len(ids) > MAX_LEN:
        SKIPPED["longer than MAX_LEN"] += 1
        return None
    token_starts = [s for s, _ in enc["offset_mapping"]]
    per_turn = sum(bisect_right(token_starts, b - 1) for _, _, b in spans)  # what one example per turn reads
    return {"input_ids": ids, "labels": labels, "task": (row.get("meta") or {}).get("family") or "other",
            "turns": len(spans), "per_turn": per_turn}

def turn_examples(row):
    """(prompt, completion) for each assistant turn: for the quick check, which asks for one turn at a time."""
    full, spans = turn_spans(row)
    return [{"prompt": p, "completion": full[a:b]} for p, a, b in spans]

def load(name):
    with open(DATA / name, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

episodes = {"train": load("sft-train.jsonl"), "val": load("sft-val.jsonl") if (DATA / "sft-val.jsonl").exists() else []}
SEEN = set(json.loads((INIT_FROM / "trained.json").read_text())) if INIT_FROM else set()
if INIT_FROM:
    new = [r for r in episodes["train"] if row_hash(r) not in SEEN]
    old = [r for r in episodes["train"] if row_hash(r) in SEEN]
    assert new, f"every episode was seen by {INIT_FROM}: nothing to update"
    replay = random.Random(SEED).sample(old, min(len(old), round(REPLAY * len(new))))
    print(f"update: {len(new)} new episodes, {len(replay)} replayed of {len(old)} seen")
    assert EXPECT_NEW is None or len(new) <= 1.5 * EXPECT_NEW, f"{len(new)} new episodes, expected about {EXPECT_NEW}"
    episodes["train"] = new + replay
TRAINED = sorted(SEEN | {row_hash(r) for r in episodes["train"]})
ds = {k: Dataset.from_list([x for r in rows if (x := episode_example(r))]) for k, rows in episodes.items()}
VAL_TURNS = [x for r in episodes["val"] for x in turn_examples(r)]
print({k: len(v) for k, v in ds.items()}, "episodes of", {k: len(v) for k, v in episodes.items()}, "|",
      sum(ds["train"]["turns"]), "training turns | skipped:", dict(SKIPPED))
assert len(ds["train"]), "no training examples"
ROWS = {"train_episodes": len(ds["train"]), "train": sum(ds["train"]["turns"]), "val": len(VAL_TURNS)}
print("families:", Counter(ds["train"]["task"]))
ex = ds["train"][0]
print(tok.decode(ex["input_ids"])[-900:])
print("--- learned in it:")
print(tok.decode([t for t in ex["labels"] if t != -100]))
n_tokens = [len(x) for x in ds["train"]["input_ids"]]
n_learned = [sum(t != -100 for t in x) for x in ds["train"]["labels"]]
print(f"tokens per episode: max {max(n_tokens)}, mean {sum(n_tokens) // len(n_tokens)}; learned "
      f"{sum(n_learned) / sum(n_tokens):.1%} of them; one example per turn would read "
      f"{sum(ds['train']['per_turn']) / sum(n_tokens):.1f}x as many tokens")

def stratified_order(tasks, seed):
    """Indices in an order where every prefix has each family in proportion to the whole."""
    rng, by_task = random.Random(seed), defaultdict(list)
    for i, t in enumerate(tasks):
        by_task[t].append(i)
    keyed = []
    for t in sorted(by_task):
        idx = by_task[t]
        rng.shuffle(idx)
        keyed += [((k + rng.random()) / len(idx), i) for k, i in enumerate(idx)]
    return [i for _, i in sorted(keyed)]

tasks = ds["train"]["task"]
ds["train"] = ds["train"].remove_columns(["task", "turns", "per_turn"])
if len(ds["val"]):
    ds["val"] = ds["val"].remove_columns(["task", "turns", "per_turn"])
else:  # a few training episodes, so the evaluation steps have something to read
    ds["val"] = ds["train"].select(range(min(4, len(ds["train"]))))
if TRAIN_EXAMPLES and TRAIN_EXAMPLES < len(tasks):
    keep = sorted(stratified_order(tasks, SEED - 1)[:TRAIN_EXAMPLES])
    ds["train"] = ds["train"].select(keep)
    tasks = [tasks[i] for i in keep]
STEPS_PER_EPOCH = math.ceil(len(ds["train"]) / (BATCH * ACCUM))
if CURVE:
    order = [i for e in range(EPOCHS) for i in stratified_order(tasks, SEED + e)]
    order = order[: max(BATCH * ACCUM, round(READ * len(order)))]
    train_stream = ds["train"].select(order)
    TOTAL_STEPS = math.ceil(len(order) / (BATCH * ACCUM))
    CURVE_STEPS = {max(1, round(f * TOTAL_STEPS)): f for f in CURVE}
    print(f"reading {len(order)} episodes ({READ:g} of {EPOCHS} epochs), {TOTAL_STEPS} optimizer steps;",
          "adapters saved at steps", sorted(CURVE_STEPS))
else:
    train_stream, CURVE_STEPS = ds["train"], {}
    TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS
print("optimizer steps:", STEPS_PER_EPOCH, "per epoch,", TOTAL_STEPS, "in all")


## Train

In [ ]:
import shutil, subprocess, sys

resume = None
if RESUME and not TRIAL:
    # only this adapter's own checkpoints: the attached earlier version (INIT_FROM) has its own
    states = [s for s in glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True)
              if f"/{ADAPTER}/checkpoints/" in s]
    if states:
        resume = str(Path(max(states, key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))).parent)
print("resume from:", resume)
if resume:  # the curve points an earlier session saved belong in this version's output too
    for done in glob.glob(f"/kaggle/input/**/{ADAPTER}/curve/read-*", recursive=True):
        shutil.copytree(done, OUT / "curve" / Path(done).name, dirs_exist_ok=True)

C = dict(MODEL=MODEL, PRECISION=PRECISION, OUT=str(OUT), ADAPTER=ADAPTER, TRIAL=TRIAL, EPOCHS=EPOCHS, READ=READ,
         CURVE_STEPS={str(k): v for k, v in CURVE_STEPS.items()}, BATCH=BATCH, ACCUM=ACCUM, PER_STEP=BATCH * ACCUM,
         LR=LR, WARMUP=WARMUP, SEED=SEED, STEPS_PER_EPOCH=STEPS_PER_EPOCH, TIME_BUDGET_H=TIME_BUDGET_H,
         SESSION_START=SESSION_START, PAD=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
         INIT_LORA=str(INIT_LORA) if INIT_FROM else None, RESUME_FROM=resume,
         STREAM="/tmp/agent-stream", VAL="/tmp/agent-val")
R = None
if DATA_PARALLEL:
    train_stream.save_to_disk(C["STREAM"])
    ds["val"].save_to_disk(C["VAL"])
    Path("/tmp/run.json").write_text(json.dumps({**C, "ARGS": {"disable_tqdm": True}}))
    proc = subprocess.Popen([sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(N_GPU),
                             "/tmp/agent_train.py", "/tmp/run.json"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() == 0:
        R = json.loads((OUT / "result.json").read_text())
    else:
        print(f"data-parallel training failed (exit {proc.returncode}, above): training with the model split "
              "across the GPUs instead")
        shutil.rmtree(OUT / "checkpoints", ignore_errors=True)
        shutil.rmtree(OUT / "curve", ignore_errors=True)
if R is None:  # one process, the model split across the GPUs (only one computes at a time)
    if model is None:
        model = agent_train.load_model(C, "auto", MAX_MEMORY, adapter=INIT_LORA if INIT_FROM else None)
    R = agent_train.train(C, model, tok, train_stream, ds["val"])
elif R["done"] or TRIAL:  # the trained adapter, split across the GPUs, for the quick check
    model = agent_train.load_model(C, "auto", MAX_MEMORY, adapter=OUT / "lora", trainable=False)

print(f"{R['minutes']:.1f} min this session, {R['processes']} process(es); peak GB per GPU {R['peak_gb']}")
DONE = R["done"]
if not DONE:  # the quick check and export would not fit in the session; the next one does them
    print(f"stopped at step {R['steps']} of {R['max_steps']}: add this version's output "
          "as an input and commit again to resume")
print("losses:", R["losses"])
print("validation:", R["validation"])

per_step = R["per_step"]
total = TOTAL_STEPS
hours = per_step * total / 3600
print(f"{per_step:.0f} s per optimizer step -> the full run ({total} steps, {EPOCHS} epochs) takes about {hours:.1f} h"
      " plus evaluation")
if hours > TIME_BUDGET_H:
    fit = int(TIME_BUDGET_H * 3600 / per_step / STEPS_PER_EPOCH)
    if CURVE:
        print(f"or set READ = {TIME_BUDGET_H * 3600 / per_step / total * READ:.2f} to fit in one")
    print(f"more than one {TIME_BUDGET_H:.0f}-hour session: commit twice (it resumes), "
          f"or set EPOCHS = {max(fit, 1)} to fit in one" + ("" if fit else " (one epoch already needs two sessions)"))


## Quick check on the validation turns

Greedy decoding of the next turn for validation turns: the same tool calls (names and arguments) as the
gold turn, or, where the gold turn is a final answer, any text and no call. A sanity check only; the
real score is `python -m nts eval --model llama --adapter <ADAPTER>` on the scenario suite.


In [ ]:
import re

model.eval()
N_CHECK = 0 if not DONE else 4 if TRIAL else 40
CALL = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)

def calls(text):
    out = []
    for body in CALL.findall(text):
        try:
            c = json.loads(body)
            out.append((c.get("name"), json.dumps(c.get("arguments", {}), sort_keys=True)))
        except json.JSONDecodeError:
            out.append(("<bad json>", body[:80]))
    return sorted(out)

score, misses = Counter(), []
for ex in VAL_TURNS[:N_CHECK]:
    inputs = tok(ex["prompt"], return_tensors="pt").to("cuda:0")  # embeddings are on GPU 0
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False, use_cache=True)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    want, got = calls(ex["completion"]), calls(text)
    kind = "call" if want else "answer"
    score[kind, "n"] += 1
    hit = got == want if want else (not got and text.strip() != "")
    score[kind, "hit"] += hit
    if want and got and {n for n, _ in got} == {n for n, _ in want}:
        score[kind, "same tool"] += 1
    if not hit:
        misses.append((want, got, text[:300]))
for kind in ("call", "answer"):
    n = score[kind, "n"]
    if n:
        print(f"{kind:6s} n={n:3d}  exact {score[kind, 'hit'] / n:.0%}"
              + (f"  same tool {score[kind, 'same tool'] / n:.0%}" if kind == "call" else ""))
for want, got, text in misses[:6]:
    print("--- want:", want, "\n    got:", got, "\n   text:", text)


## Export

1. the PEFT adapter (`lora/`), `trained.json` (the rows it has read, for an update with
   `INIT_FROM`) and `manifest.json` (adapter, base, tasks, parent, rows);
2. the adapter as a **LoRA GGUF** (`gguf/<ADAPTER>.gguf`, f16, tens of MB) with llama.cpp's
   `convert_lora_to_gguf.py`, against the base's own config; with a curve, one per point;
3. with `MERGE = True` only: the adapter merged into the bf16 base and quantized to Q8_0, as
   before (a whole model per adapter, for a server that cannot load adapters).

A TRIAL exports 1 and 2 too, from its 20 steps, to check conversion and serving early.


In [ ]:
import gc

if not DONE and not TRIAL:
    print("training not finished: no export this session")
else:
    model.save_pretrained(str(OUT / "lora"))
    tok.save_pretrained(str(OUT / "lora"))
    (OUT / "trained.json").write_text(json.dumps(TRAINED))
    MANIFEST = {"adapter": ADAPTER, "base": MODEL, "tasks": TASKS,
                "parent": json.loads((INIT_FROM / "manifest.json").read_text())["adapter"] if INIT_FROM else None,
                "rows": ROWS, "epochs": EPOCHS, "steps": R["steps"], "processes": R["processes"], "trial": TRIAL,
                "lr": LR, "max_len": MAX_LEN}
    # the data it read, as training.dataset recorded it: recipe, code version and every split's fingerprint
    if (DATA / "compile-manifest.json").exists():
        data = json.loads((DATA / "compile-manifest.json").read_text())
        MANIFEST["data"] = {k: data.get(k) for k in ("recipe", "recipe_sha", "code", "schema", "fingerprints")}
    (OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))
    print(json.dumps(MANIFEST, indent=1))
    del model
    gc.collect()
    torch.cuda.empty_cache()

    from peft import PeftModel
    from transformers import AutoProcessor

    def merge(adapter, out):
        """The adapter merged into the original bf16 weights, on the CPU."""
        shutil.rmtree(out, ignore_errors=True)
        base = AutoModelForImageTextToText.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cpu")
        merged = PeftModel.from_pretrained(base, str(adapter)).merge_and_unload()
        merged.save_pretrained(out)
        AutoProcessor.from_pretrained(MODEL).save_pretrained(out)
        tok.save_pretrained(out)
        del base, merged
        gc.collect()

    # what to export (file name, adapter): each curve point or the one adapter, and the earlier version this
    # one was updated from (its GGUF serves the tasks this one was not trained on)
    POINTS = ([(f"{ADAPTER}-{p.name}", p) for p in sorted((OUT / "curve").glob("read-*"))] if CURVE and not TRIAL else
              [(ADAPTER, OUT / "lora")])
    if INIT_FROM and not TRIAL:
        POINTS.append((MANIFEST["parent"], INIT_LORA))
    print("export:", [n for n, _ in POINTS])


In [ ]:
if DONE or TRIAL:
    from huggingface_hub import snapshot_download

    (OUT / "gguf").mkdir(parents=True, exist_ok=True)
    # the server's build (b11368): a newer converter can write a GGUF the server cannot read
    !git clone --depth 1 --branch b11368 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
    !pip install -q /tmp/llama.cpp/gguf-py sentencepiece
    BASE_DIR = snapshot_download(MODEL)  # the base's config and tokenizer (already downloaded)
    for tag, adapter in POINTS:
        !python /tmp/llama.cpp/convert_lora_to_gguf.py {adapter} --base {BASE_DIR} --outtype f16 --outfile {OUT}/gguf/{tag}.gguf
    if MERGE and not TRIAL:
        !cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
        !cmake --build /tmp/llama.cpp/build --target llama-quantize -j 4 > /dev/null
        MERGED = Path("/tmp/merged")
        for tag, adapter in POINTS[:-1] if INIT_FROM else POINTS:  # not the parent (appended last)
            merge(adapter, MERGED)
            # --no-mtp: the converter counts Qwen3.5's MTP layer in block_count, but the merged
            # checkpoint has no mtp.* weights (transformers drops them), so blk.32 would be missing
            !python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --no-mtp --outtype bf16 --outfile /tmp/model-bf16.gguf
            !/tmp/llama.cpp/build/bin/llama-quantize /tmp/model-bf16.gguf {OUT}/gguf/{tag}-merged.Q8_0.gguf Q8_0
            !rm -f /tmp/model-bf16.gguf
    for f in sorted(Path("/kaggle/working").rglob("*.gguf")):
        print(f, f"{f.stat().st_size / 2**20:.0f} MB")


## Check the GGUFs

Each exported model must have weights for every layer its header counts, as `llama-server` checks on load. Qwen3.5's config counts an MTP layer the fine-tuned checkpoint does not have, and a converter that counts it writes a file the server rejects ("tensor 'blk.32.attn_norm.weight' not found"). Such a file is renamed `*.broken` and the run fails here instead of on the server.

In [ ]:
def check_gguf(path):
    """Every layer the GGUF's header counts has weights, and none beyond. A converter can count a layer the
    checkpoint does not have (Qwen3.5's MTP head, which transformers drops: the server then fails with
    "tensor 'blk.32.attn_norm.weight' not found"). Such a file is renamed *.broken so it is not downloaded."""
    from gguf import GGUFReader
    reader = GGUFReader(str(path))

    def field(key, default=None):
        f = reader.fields.get(key)
        return default if f is None else f.contents()

    arch = field("general.architecture")
    n, n_mtp = field(f"{arch}.block_count"), field(f"{arch}.nextn_predict_layers", 0)
    if n is None:  # a LoRA adapter has no layer count of its own
        print(f"{Path(path).name}: adapter, {len(reader.tensors)} tensors")
        return True
    layers = {int(t.name.split(".")[1]) for t in reader.tensors if t.name.startswith("blk.")}
    missing, extra = sorted(set(range(n)) - layers), sorted(layers - set(range(n)))
    del reader  # it maps the file, which cannot be renamed while mapped on Windows
    if not missing and not extra:
        print(f"{Path(path).name}: {n} layers, all with weights")
        return True
    broken = Path(str(path) + ".broken")
    Path(path).rename(broken)
    print(f"BROKEN {Path(path).name}: the header counts {n} layers ({n_mtp} of them MTP) but has no weights "
          f"for {missing} (and weights beyond the count for {extra}); renamed to {broken.name}. "
          "Convert with --no-mtp, with the llama.cpp the server runs.")
    return False


if DONE or TRIAL:
    ok = [check_gguf(f) for f in sorted((OUT / "gguf").glob("*.gguf"))]
    assert all(ok), "a GGUF the server cannot load (see BROKEN above)"


## Next: serve it

Download `gguf/<ADAPTER>.gguf` from the version's *Output* tab (keep `lora/`, `trained.json`
and `manifest.json` too: the next update starts from them). On the server, one base and every
adapter, none applied until a request asks for it:

```sh
llama-server -m Qwen3.5-4B-Base-Q8_0.gguf --lora nts-intents.gguf,nts-policy.gguf,nts-negotiation.gguf \
    --lora-init-without-apply --jinja -ngl 99 -c 8192 --port 8080
```

`curl <server>/lora-adapters` lists them. Name the adapter in `adapters.toml` and point the
project at it: `NTS_ADAPTERS=adapters.toml` for the web app, `--adapters adapters.toml` (or
`--adapter <name>` for one task) for the `--local` evaluations. Each request switches on its own
adapter (the `lora` field of the request). Thinking stays off, as in training. A server that cannot
take per-request adapters can serve each one as a model of its own name instead (a router preset of
the base with `lora = <file>`, or the `MERGE` GGUF): the client finds it by the same name.


For the agent adapter: add `nts-agent-v1.gguf` to the server's `--lora` list (with `--jinja`, which
tool calls need), then from the agentic worktree:

```sh
python -m nts eval --model llama --adapter nts-agent-v1 --per-family 2 --seed 0
```
